### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="wids_diabetes_mellitus",
    dataset_year="2021",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/widsdatathon2021",
    download_description="""
We download the data from Kaggle.

kaggle competitions download -c widsdatathon2021 -f TrainingWiDS2021.csv && unzip TrainingWiDS2021.csv.zip &&  rm TrainingWiDS2021.csv.zip
mkdir -p local-data-warehouse/wids_diabetes_mellitus && mv TrainingWiDS2021.csv local-data-warehouse/wids_diabetes_mellitus/
""",
    # References
    academic_reference_bibtex=r"""@misc{Matthys2021WiDSDatathon2021,
  author = {Karen Matthys and Meredith Lee and Neha Goel and Sharada Kalanidhi and Valerie and Vani M.},
  title  = {WiDS Datathon 2021},
  year   = {2021},
  howpublished = {\url{https://kaggle.com/competitions/widsdatathon2021}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Matthys2021WiDSDatathon2021",
    license="Kaggle Competition Rules",
    data_tags=["IID"],
    curation_comments="""
We start with the Kaggle dataset.

- Data from the prior year's hackathon (2020) also used the very similar data but with older data and different splits. But it seems the feature engineering was transferable across years.
- We drop encounter_id as it is an identifier with no temporal information, and from the competition there seems to be no temporal or index-based leakage.
- We follow the top solutions and use normal random splits, instead of stratifying by hospital_id or ICU ID. Moreover, with the data for which we have labels, we have an even larger overlap for hospital_id and ICU ID even more than the train-test split on Kaggle. Lastly, we do not need to decode the hospital_ID based on the ICU ID.
- A hospital can have multiple ICUs, so there is a strong co-correlation between hospital_id and ICU ID, but it is not identical.
- We drop rows with age=0, as these seems to be 30 faulty entries (patients have a large height at age 0).
- Kaggle experts found that several features were likely clipped to a min/max value based on investigating the histograms of the data. We noticed this too. Following the experts, we add a flag for these features indicating if the value was clipped or not.
- Several Kaggle experts also re-computed the BMI. We also noticed that for 2770 rows, the BMI did not match the real BMI based on the weight and height. We drop all of these cases, as it is unclear to us what the cause of this error is. Given the small fraction, there is either a data entry error, or some values got corrected manually.
- We found and dropped three duplicated columns that provided no new information.
- We keep medical codes as string, as these could be more meaningfully decoded.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="diabetes_mellitus",
    problem_type="binary_classification",
    objective_metric_name="roc_auc",
    stratify_on="diabetes_mellitus",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "TrainingWiDS2021.csv")
print("Loaded data shape:", df.shape)

df = df.drop(columns=[
    "Unnamed: 0", # Saving artifact
    "encounter_id", # Identifier, no temporal information
    "readmission_status", # Constant
    # Duplicated columns (no new information)
    "paco2_for_ph_apache",
    "h1_inr_max",
    "h1_inr_min",
])

df = df[df["age"] != 0]  # Drop rows with age=0 (faulty entries)

# Drop cases with faulty BMI calculations
df["bmi_new"] = df["weight"] / ((df["height"] / 100) ** 2)
bmi_wrong_mask = (df["bmi"].round(2) != df["bmi_new"].round(2)) & ~df["bmi"].isna()
df = df[~bmi_wrong_mask]
df = df.drop(columns=["bmi_new"])

# add min/max flags for features that were clipped
clipped_features = [
    "d1_sysbp_min",
    "d1_sysbp_max",
    "d1_heartrate_min",
    "d1_heartrate_max",
    "d1_diasbp_min",
    "d1_diasbp_max",
    "heart_rate_apache",
]
new_clipped_features = []
for f in clipped_features:
    min_val = df[f].min()
    max_val = df[f].max()
    clip_f = f + "_clipped"
    new_clipped_features.append(clip_f)
    df[clip_f] = ((df[f] == min_val) | (df[f] == max_val))

as_cat_type = [
    "hospital_id",
    "icu_id",
    "elective_surgery",
    "ethnicity",
    "gender",
    "hospital_admit_source",
    "icu_admit_source",
    "icu_stay_type",
    "icu_type",
    "apache_post_operative",
    "arf_apache",
    "intubated_apache",
    "gcs_unable_apache",
    # We keep medical scales as int (as they are ordinal in nature)
    "ventilated_apache",
    "aids",
    "cirrhosis",
    "hepatic_failure",
    "immunosuppression",
    "leukemia",
    "lymphoma",
    "solid_tumor_with_metastasis",
    "diabetes_mellitus",
] + new_clipped_features
as_string_type = [
    # Codes for medical terms
    "apache_2_diagnosis",
    "apache_3j_diagnosis",
]
for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")
df[as_cat_type] = df[as_cat_type].astype("category")

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

Loaded data shape: (130157, 181)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 127,358
Columns: 182
Use sampling: False (sample size: 127,358)


Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['bmi', 'urineoutput_apache', 'pre_icu_los_days', 'd1_pao2fio2ratio_max', 'd1_pao2fio2ratio_min', 'weight', 'h1_pao2fio2ratio_max', 'h1_pao2fio2ratio_min', 'd1_wbc_max', 'wbc_apache']
Rows remaining as candidates after top-10 filter: 358 (of 127,358)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,hospital_id,age,bmi,elective_surgery,ethnicity,gender,height,hospital_admit_source,icu_admit_source,icu_id,icu_stay_type,icu_type,pre_icu_los_days,weight,albumin_apache,apache_2_diagnosis,apache_3j_diagnosis,apache_post_operative,arf_apache,bilirubin_apache,bun_apache,creatinine_apache,fio2_apache,gcs_eyes_apache,gcs_motor_apache,gcs_unable_apache,gcs_verbal_apache,glucose_apache,heart_rate_apache,hematocrit_apache,intubated_apache,map_apache,paco2_apache,pao2_apache,ph_apache,resprate_apache,sodium_apache,temp_apache,urineoutput_apache,ventilated_apache,wbc_apache,d1_diasbp_invasive_max,d1_diasbp_invasive_min,d1_diasbp_max,d1_diasbp_min,d1_diasbp_noninvasive_max,d1_diasbp_noninvasive_min,d1_heartrate_max,d1_heartrate_min,d1_mbp_invasive_max,d1_mbp_invasive_min,d1_mbp_max,d1_mbp_min,d1_mbp_noninvasive_max,d1_mbp_noninvasive_min,d1_resprate_max,d1_resprate_min,d1_spo2_max,d1_spo2_min,d1_sysbp_invasive_max,d1_sysbp_invasive_min,d1_sysbp_max,d1_sysbp_min,d1_sysbp_noninvasive_max,d1_sysbp_noninvasive_min,d1_temp_max,d1_temp_min,h1_diasbp_invasive_max,h1_diasbp_invasive_min,h1_diasbp_max,h1_diasbp_min,h1_diasbp_noninvasive_max,h1_diasbp_noninvasive_min,h1_heartrate_max,h1_heartrate_min,h1_mbp_invasive_max,h1_mbp_invasive_min,h1_mbp_max,h1_mbp_min,h1_mbp_noninvasive_max,h1_mbp_noninvasive_min,h1_resprate_max,h1_resprate_min,h1_spo2_max,h1_spo2_min,h1_sysbp_invasive_max,h1_sysbp_invasive_min,h1_sysbp_max,h1_sysbp_min,h1_sysbp_noninvasive_max,h1_sysbp_noninvasive_min,h1_temp_max,h1_temp_min,d1_albumin_max,d1_albumin_min,d1_bilirubin_max,d1_bilirubin_min,d1_bun_max,d1_bun_min,d1_calcium_max,d1_calcium_min,d1_creatinine_max,d1_creatinine_min,d1_glucose_max,d1_glucose_min,d1_hco3_max,d1_hco3_min,d1_hemaglobin_max,d1_hemaglobin_min,d1_hematocrit_max,d1_hematocrit_min,d1_inr_max,d1_inr_min,d1_lactate_max,d1_lactate_min,d1_platelets_max,d1_platelets_min,d1_potassium_max,d1_potassium_min,d1_sodium_max,d1_sodium_min,d1_wbc_max,d1_wbc_min,h1_albumin_max,h1_albumin_min,h1_bilirubin_max,h1_bilirubin_min,h1_bun_max,h1_bun_min,h1_calcium_max,h1_calcium_min,h1_creatinine_max,h1_creatinine_min,h1_glucose_max,h1_glucose_min,h1_hco3_max,h1_hco3_min,h1_hemaglobin_max,h1_hemaglobin_min,h1_hematocrit_max,h1_hematocrit_min,h1_lactate_max,h1_lactate_min,h1_platelets_max,h1_platelets_min,h1_potassium_max,h1_potassium_min,h1_sodium_max,h1_sodium_min,h1_wbc_max,h1_wbc_min,d1_arterial_pco2_max,d1_arterial_pco2_min,d1_arterial_ph_max,d1_arterial_ph_min,d1_arterial_po2_max,d1_arterial_po2_min,d1_pao2fio2ratio_max,d1_pao2fio2ratio_min,h1_arterial_pco2_max,h1_arterial_pco2_min,h1_arterial_ph_max,h1_arterial_ph_min,h1_arterial_po2_max,h1_arterial_po2_min,h1_pao2fio2ratio_max,h1_pao2fio2ratio_min,aids,cirrhosis,hepatic_failure,immunosuppression,leukemia,lymphoma,solid_tumor_with_metastasis,diabetes_mellitus,d1_sysbp_min_clipped,d1_sysbp_max_clipped,d1_heartrate_min_clipped,d1_heartrate_max_clipped,d1_diasbp_min_clipped,d1_diasbp_max_clipped,heart_rate_apache_clipped
0,118,86.0,26.488696,1,Caucasian,M,175.3,Operating Room,Operating Room / Recovery,92,admit,CTICU,0.002778,81.40,NaN,203.0,1206.03,1,0,NaN,28.0,1.00,0.48,4.0,6.0,0.0,5.0,132.0,66.0,34.0,0,112.0,37.0,184.0,7.35,26.0,137.0,36.2,NaN,0,6.80,76.0,50.0,71.0,71.0,71.0,71.0,94.0,66.0,118.0,72.0,118.0,72.0,NaN,NaN,27.0,16.0,100.0,97.0,170.0,126.0,158.0,158.0,158.0,158.0,36.8,36.2,62.0,58.0,62.0,58.0,NaN,NaN,74.0,68.0,90.0,82.0,90.0,82.0,NaN,NaN,19.0,16.0,100.0,99.0,144.0,136.0,144.0,136.0,NaN,NaN,36.2,36.2,NaN,NaN,NaN,NaN,28.0,28.0,7.5,7.5,1.00,1.00,132.0,132.0,22.0,22.0,11.6,11.5,34.1,34.0,NaN,NaN,NaN,NaN,137.0,131.0,4.9,4.7,137.0,137.0,7.50,6.80,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.6,11.6,34.0,34.0,NaN,NaN,137.0,137.0,NaN,NaN,NaN,NaN,7.5,7.5,37.0,37.0,7.35,7.35,184.0,184.0,383.333333,383.333333,37.0,37.0,7.35,7.35,184.0,184.0,383.333333,383.333333,0,0,0,0,0,0,0,0,False,False,False,False,False,False,False
1,30,30.0,25.850545,0,Caucasian,M,172.7,Emergency Department,Accident & Emergency,921,

In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,hospital_admit_source,category,32415.0,25.45,15.0,"Emergency Department, Operating Room, Floor, Direct Admit, Recovery Room, Other Hospital, Acute Care/Floor, Step-Down Unit (SDU), PACU, Other ICU"
1,ethnicity,category,1533.0,1.20,6.0,"Caucasian, African American, Other/Unknown, Hispanic, Asian, Native American"
2,gcs_unable_apache,category,681.0,0.53,2.0,"0.0, 1.0"
3,icu_admit_source,category,221.0,0.17,5.0,"Accident & Emergency, Operating Room / Recovery, Floor, Other Hospital, Other ICU"
4,gender,category,40.0,0.03,2.0,"M, F"
5,hospital_id,category,0.0,0.00,204.0,"118, 19, 188, 86, 7, 161, 175, 70, 196, 176"
6,elective_surgery,category,0.0,0.00,2.0,"0, 1"
7,icu_id,category,0.0,0.00,328.0,"1019, 646, 876, 653, 413, 998, 236, 337, 1012, 1078"
8,icu_stay_type,category,0.0,0.00,3.0,"admit, transfer, readmit"
9,icu_type,category,0.0,0.00,8.0,"Med-Surg ICU, CCU-CTICU, MICU, Neuro ICU, SICU, Cardiac ICU, CSICU, CTICU"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,122489.0,62.088081,16.772879,16.000000,89.000000
bmi,122886.0,28.904763,7.559969,14.844926,67.812636
height,125298.0,169.695366,10.495597,137.200000,195.590000
pre_icu_los_days,127358.0,0.839402,2.487524,-0.250000,175.627778
weight,123911.0,83.458102,23.525208,38.600000,186.000000
albumin_apache,50803.0,2.888710,0.689554,1.200000,4.600000
bilirubin_apache,46512.0,1.202893,2.357805,0.100000,60.200000
bun_apache,102511.0,25.675660,20.652158,4.000000,127.000000
creatinine_apache,103040.0,1.480846,1.541544,0.300000,11.180000
fio2_apache,29717.0,0.595963,0.262634,0.210000,1.000000


In [7]:
# Categorical Feature Statistics
cat_stats

value   count    pct
column                      rank                                          
aids                        1                             0  127228   99.9
                            2                             1     130    0.1
apache_2_diagnosis          1                         113.0   15912  12.49
                            2                         301.0    9676    7.6
                            3                         302.0    8740   6.86
                            4                         112.0    5899   4.63
                            5                         308.0    5773   4.53
apache_3j_diagnosis         1                        501.05    5991    4.7
                            2                        107.01    5899   4.63
                            3                        403.01    5329   4.18
                            4                        106.01    5190   4.08
                            5                        703.03    4129   3.24
apache_post_operative       1                             0  100798  79.15
                            2                             1   26560  20.85
arf_apache                  1                             0  123810  97.21
                            2                             1    3548   2.79
cirrhosis                   1                             0  125294  98.38
                            2                             1    2064   1.62
d1_diasbp_max_clipped       1                         False  125997  98.93
                            2                          True    1361   1.07
d1_diasbp_min_clipped       1                         False  125970  98.91
                            2                          True    1388   1.09
d1_heartrate_max_clipped    1                         False  125944  98.89
                            2                          True    1414   1.11
d1_heartrate_min_clipped    1                         False  126483  99.31
                            2                          True     875   0.69
d1_sysbp_max_clipped        1                         False  126057  98.98
                            2                          True    1301   1.02
d1_sysbp_min_clipped        1                         False  126050  98.97
                            2                          True    1308   1.03
diabetes_mellitus           1                             0   99801  78.36
                            2                             1   27557  21.64
elective_surgery            1                             0  103005  80.88
                            2                             1   24353  19.12
ethnicity                   1                     Caucasian   98171  77.08
                            2              African American   13538  10.63
                            3                 Other/Unknown    6131   4.81
                            4                      Hispanic    4957   3.89
                            5                         Asian    2136   1.68
gcs_unable_apache           1                           0.0  125232  98.33
                            2                           1.0    1445   1.13
                            3                          <NA>     681   0.53
gender                      1                             M   69152   54.3
                            2                             F   58166  45.67
                            3                          <NA>      40   0.03
heart_rate_apache_clipped   1                         False  126025  98.95
                            2                          True    1333   1.05
hepatic_failure             1                             0  125613  98.63
                            2                             1    1745   1.37
hospital_admit_source       1          Emergency Department   50082  39.32
                            2                          <NA>   32415  25.45
                            3                Operating Room   13585  10.67
    

In [8]:
# Target Distribution
target_df

,count,pct
diabetes_mellitus,,
0,99801,78.36
1,27557,21.64


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...


Saving curated container to wids_diabetes_mellitus/019d5dca-97d8-77e8-9938-c7284735aaed


019d5dca-97d8-77e8-9938-c7284735aaed
04e0480c1b15a6df6539af6260d9fc85e49e03c8d659ec0a16dffe6958df1deb
